In [ ]:
import sys
import os

# Aggiungi project root al path per gli import
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project Root aggiunta al path: {project_root}")

In [ ]:
# CALCOLO NAIVE MAE PER I 3 FOLD DI VALIDAZIONE
import pandas as pd
import torch
import torch.nn as nn
from src.ModelClasses.naive import NaivePersistence
from src.Training.engine import validate_one_epoch
from src.DataLoading.data_loader import TS_Cross_Validator
from src.config import TARGET_COL, SAMPLING_CONFIG, NAIVE_CONFIG

# 1. Caricamento Dati 
df = pd.read_csv("../data/processed/preprocessed_ds.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COL}")

# 2. Creazione Folds
validator = TS_Cross_Validator(df, TARGET_COL, SAMPLING_CONFIG)
folds = list(validator.get_folds())  # Converti generator in lista

# 3. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Usiamo L1Loss (MAE) per il denominatore del MASE
mae_metric = nn.L1Loss()

naive_maes = []

print(f"\n--- Calcolo Benchmark Naive su {len(folds)} Fold ---\n")

# Crea modello Naive con model_config (non usa più train_loader)
naive_model = NaivePersistence(model_config=NAIVE_CONFIG).to(device)

for i, (train_loader, val_loader, scaler) in enumerate(folds):
    # validate_one_epoch restituisce 3 valori: (avg_loss, avg_mae, avg_rmse)
    _, fold_mae, _ = validate_one_epoch(naive_model, val_loader, nn.MSELoss(), device)
    
    naive_maes.append(fold_mae)
    print(f"FOLD {i+1} -> Naive MAE: {fold_mae:.6f}")

print("\n--- COPIA QUESTO NEL TUO training_config.py ---")
print(f"NAIVE_MAE_PER_FOLD = {naive_maes}")

In [ ]:
# CALCOLO NAIVE MAE PER FINAL FOLD (22 mesi train + 2 mesi val)

from src.DataLoading import create_final_train_val_loaders
# Crea train/val loaders per il final fold
_, final_val_loader, _ = create_final_train_val_loaders(
    df, target_col=TARGET_COL
)
# Calcola MAE del modello naive sul final validation fold
_, final_fold_mae, _ = validate_one_epoch(
    naive_model, final_val_loader, nn.MSELoss(), device
)
print(f"\n--- FINAL FOLD (22 mesi train + 2 mesi val) ---")
print(f"Naive MAE: {final_fold_mae:.6f}")
print("\n--- COPIA QUESTO IN src/config/training_config.py ---")
print(f"NAIVE_MAE_FINAL_FOLD = {final_fold_mae}")